In [1]:
df = spark.read.table("mtomactual") 
dfdates = spark.read.table("mtomactualwithdates") 
dfadditional = spark.read.table("mtomactualadditional") 
display(df) 
display(dfadditional) 
display(dfdates) 


StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 72e16f44-dd83-4350-bd1d-b9710913dde7)

SynapseWidget(Synapse.DataFrame, 1f4a20ed-7b56-4dab-84c9-e17a060a2967)

SynapseWidget(Synapse.DataFrame, 2d073ac9-d93f-45e5-84c5-44e0b97e85ce)

In [2]:
dfunion = df.union(dfadditional) 
display(dfunion) 

dfunion.write.mode("overwrite").format("delta").saveAsTable("MtoMActualCombined") 

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0a7f949e-f195-4d30-8a01-6564d3ada24f)

In [3]:
dfunion = df.unionByName(dfdates, allowMissingColumns=True) 
display(dfunion)

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2ff94c08-7d34-4165-9ee4-e11088a1af2c)

In [4]:
%%sql 
SELECT * 
FROM mtomactual 
UNION ALL -- OR UNION ALL 
SELECT * 
FROM mtomactualadditional

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 6, Finished, Available, Finished, False)

<Spark SQL result set with 9 rows and 3 fields>

In [5]:
%%sql 
SELECT Country, Location, Actual, NULL as ColDate 
FROM mtomactual 
UNION ALL 
SELECT Country, Location, Actual, ColDate 
FROM mtomactualwithdates 

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 7, Finished, Available, Finished, False)

<Spark SQL result set with 12 rows and 4 fields>

In [6]:
display(df.groupBy("Country","Location","Actual").count())

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 74148f7f-0585-4b30-994b-92b769780daf)

In [7]:
%%sql 
SELECT Country, Location, Actual 
FROM mtomactualcombined 
GROUP BY Country, Location, Actual 
HAVING COUNT(*)>1 

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 9, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [8]:
%%sql 
SELECT DISTINCT Country, Location, Actual 
FROM mtomactualcombined 

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 10, Finished, Available, Finished, False)

<Spark SQL result set with 8 rows and 3 fields>

In [9]:
df = spark.read.table("mtomactualcombined") 
display(df.distinct()) 

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6eb627d9-9065-463c-a078-94c2245c98d5)

In [10]:
dfactual = spark.read.table("mtomactual") 
dftarget = spark.read.format("csv").option("header","true").load("Files/MtoMTarget.csv") 
dftarget.write.mode("overwrite").format("delta").saveAsTable("MtoMTarget") 
display(dfactual) 
display(dftarget)

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 50a664e5-70ff-4715-93d3-dce32f6fafd6)

SynapseWidget(Synapse.DataFrame, 88d07068-641e-4c34-b494-0cd27d5ca93e)

In [11]:
dfactual = dfactual.select(dfactual.Country, dfactual.Actual.cast("int"))\
                   .groupBy("Country").sum("Actual").withColumnRenamed("sum(Actual)","ActualTotal") 
display(dfactual)

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d1518f46-7520-48b9-b27e-e86948faf518)

In [12]:
dftarget = dftarget.select(dftarget.Country, dftarget.Target.cast("int")).groupBy("Country").sum("Target").withColumnRenamed("sum(Target)","TargetTotal") 
display(dftarget)

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b194aff5-8f80-4398-b039-cf02a43f015b)

In [13]:
dfactual.write.mode("overwrite").format("delta").saveAsTable("MtoMActualSum") 
dftarget.write.mode("overwrite").format("delta").saveAsTable("MtoMTargetSum")

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 15, Finished, Available, Finished, False)

In [14]:
dfjoin = dfactual.join(dftarget, dfactual.Country == dftarget.Country) 
display(dfjoin) 
display(dfactual.join(dftarget, "Country")) 

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1da344ed-bd25-4d3c-909a-367e99d5ef46)

SynapseWidget(Synapse.DataFrame, c3898f30-3689-453f-8be4-6b8802e2cec6)

In [15]:
%%sql 
SELECT MtoMActualSum.Country, ActualTotal,  
       MtoMTargetSum.Country, TargetTotal 
FROM MtoMActualSum
FULL JOIN MtoMTargetSum 
ON MtoMActualSum.Country = MtoMTargetSum.Country 

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 17, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 4 fields>

In [16]:
dfactual = dfactual.withColumnRenamed("Country", "ActualCountry") 
dftarget = dftarget.withColumnRenamed("Country", "TargetCountry") 
dfjoin = dfactual.join(dftarget, dfactual.ActualCountry == dftarget.TargetCountry, "full") 
display(dfjoin) 
dfjoin.write.mode("overwrite").format("delta").saveAsTable("MtoMJoin")

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 40ea3852-c6e8-4bab-ba05-46ab0c10f016)

In [17]:
%%sql 
SELECT MtoMActualSum.Country AS ActualCountry, ActualTotal,  
       MtoMTargetSum.Country AS TargetCountry, TargetTotal 
FROM MtoMActualSum 
FULL JOIN MtoMTargetSum 
ON MtoMActualSum.Country = MtoMTargetSum.Country 


StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 19, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 4 fields>

In [18]:
display(dfjoin.where(dfjoin.ActualTotal.isNull()))

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 66817b53-2464-4828-8b92-8254f15970e7)

In [19]:
%%sql
SELECT MtoMActualSum.Country AS ActualCountry, ActualTotal,  
       MtoMTargetSum.Country AS TargetCountry, TargetTotal 
FROM MtoMActualSum 
FULL JOIN MtoMTargetSum 
ON MtoMActualSum.Country = MtoMTargetSum.Country 
WHERE ActualTotal IS NULL 

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 21, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [22]:
dfjoin = dfjoin.fillna({"ActualCountry":"(No Actual)"}) 
dfjoin = dfjoin.fillna({"TargetCountry":"(No Target)"}) 
dfjoin = dfjoin.na.fill(0)
display(dfjoin)

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8eb43043-9285-42ee-ac52-7375bdc09bd1)

In [23]:
%%sql

SELECT IFNULL(MtoMActualSum.Country, "(No Actual)") AS ActualCountry, COALESCE(ActualTotal, 0) AS ActualTotal, 
       IFNULL(MtoMTargetSum.Country, "(No Target)") AS TargetCountry, COALESCE(TargetTotal, 0) AS TargetTotal

FROM MtoMActualSum 
FULL JOIN MtoMTargetSum 
ON MtoMActualSum.Country = MtoMTargetSum.Country 

StatementMeta(, 49fe034f-1637-4608-8b70-3c58fe01d288, 25, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 4 fields>